# Finite-segment Galerkin experiments with discontinuous materials

Solve a BSK model or check a problem with a known answer on **0 ≤ x ≤ R**.
Choose **1, 2, 3, 5, 7, or 10 interfaces**, or enter your own material table.
Start with **Run All**: the main example and two known-answer checks run by
default. Larger studies, high precision, homotopy comparisons, and file export
each have their own switch, initially `False`.

Install once with `python -m pip install -e ".[test,notebook]"` from the project
root. Select that environment as this notebook's kernel. No directory change
is needed. Restart the kernel after changing installed source code.

## What is being solved?

$$
 [\epsilon(x)(u'-\delta^2u''')]'=f(x,u),\qquad 0<x<R,
$$
$$
 u(0)=u_{\rm L},\quad u'''(0)=q_{\rm L},\qquad
 u(R)=u_{\rm R},\quad u'(R)=s_{\rm R}.
$$

The material coefficient epsilon must stay positive. Its interfaces may have
jumps. The solver also approximates **w = −u″** independently and reports
the flux **J = epsilon (u′ + delta² w′)**. No condition sets w(R) to zero.

The two potential values are built into the approximation. The left third
derivative and right slope enter as weak boundary loads; their pointwise
numerical derivatives need not equal the prescribed values exactly at finite N.

**R sets the physical interval. N sets spatial resolution:** N−1 potential
coefficients and N+1 auxiliary coefficients give **2N unknowns**. Arithmetic
precision and Newton's absolute residual tolerance are separate settings.
There is no half-line mapping length L in this example.

In [ ]:
%matplotlib inline
from pathlib import Path
import os
from matplotlib import pyplot as plt
from overscreening_breakdown import (
    BSKProblem, FluxProblem, LayeredMaterial, Coefficient,
    ContinuationOptions, ScreenedLinearReference,
)
from overscreening_breakdown.manufactured import finite_segment_case
from overscreening_breakdown.notebook_examples.discontinuous_galerkin import (
    run_defined_problem, report_experiment, report_error_table,
    report_convergence_table, export_experiment,
    homotopy_stage_rows, report_homotopy_stages, report_homotopy_comparison,
)
from overscreening_breakdown.notebook_examples.segment_galerkin import (
    segment_material, segment_material_table,
)
from overscreening_breakdown.notebook_examples.discontinuous_plotting import (
    plot_material, plot_material_gallery, plot_experiment, plot_errors,
    plot_convergence, plot_domain_comparison,
)
from overscreening_breakdown.notebook_examples.homotopy_plotting import (
    plot_homotopy_path, plot_homotopy_comparison,
)


## 1. Your physical experiment — edit this cell

Use decimal strings such as `"0.1"` to retain the intended digits in mpmath.
Boundary rows below list derivative orders 0 through 4; `None` means that
condition is unspecified. The BSK parameters retain the project's existing
physical-source convention. Coordinates and fields use the model's dimensionless
units, with no automatic conversion from metres or volts. The Gaussian term
retains the existing convention without an additional 1/sigma prefactor.

Presets put interfaces at equal fractions of R. A custom table lists
`(start_position, epsilon_value)` rows, beginning at zero, with every later
start strictly below R. The last row ends at R in this experiment.
`"graded"` materials vary smoothly inside each layer and jump at interfaces.

In [ ]:
R = "4"
interface_count = 3                 # 1, 2, 3, 5, 7, or 10
material_style = "constant"         # "constant" or "graded"
layers = None                       # None selects the preset
# Example custom table for R=4:
# layers = [("0", "1"), ("1", "2"), ("2.5", "0.8")]

model = "BSK"                       # "BSK", "manufactured_linear", "manufactured_nonlinear"
delta = "2"
wall_potential = "1"
wall_third_derivative = "0"
right_potential = "6"
right_slope = "0"
gamma = "0.5"
sigma = "2"
mu = "0.1"
beta_nonlinear = "0.125"

accuracy = "standard"               # "standard", "strict_integration", "high_precision"
verbosity = 1                       # 0 quiet; 1 summaries; 2 iterations; 3 trials
progress_callback = None            # Optional function receiving progress events
progress_interval = 1.0
numerical_settings = dict(progress_callback=progress_callback,
                          progress_interval=progress_interval)

# Display only: x_min="1", x_max="3" zooms within R=4; None shows the full actual R.
plot_x_min = "0"
plot_x_max = None  # None uses each result's actual R; an explicit bound must fit its domain.

# History axes count Newton iterations and quadrature orders, not physical x.
iteration_xlim = None       # Example: (0, 50)
quadrature_xlim = None      # Example: (0, 128)


## 2. Check the material and interval

Check the printed endpoints and coefficient values before solving. In the
gallery later on, all six interface counts fit inside the same finite interval.
The solver requires the interface locations and positive bounds, but it does
not require a derivative of epsilon.

In [ ]:
material = (segment_material(interface_count, R=R, kind=material_style) if layers is None
            else LayeredMaterial(layers, kind=material_style))
print(segment_material_table(material, R=R, accuracy=accuracy))
material_figure = plot_material(material, accuracy=accuracy, x_min=plot_x_min, x_max=R if plot_x_max is None else plot_x_max)
plt.show()


## 3a. Define the BSK problem explicitly

This constructor supplies the physical parameters, finite domain, four
boundary conditions, and starting field. Edit the definition here to change
the experiment. `run_defined_problem` below solves this object as supplied.

In [ ]:
def define_bsk_problem(epsilon, *, R, wall_potential, wall_third_derivative,
                       right_potential, right_slope, delta, gamma, sigma, mu):
    def initial_guess(x, a):
        t = x / a.scalar(R)
        return a.scalar(wall_potential) * (1 - t) + a.scalar(right_potential) * t

    return BSKProblem(
        delta=delta, gamma=gamma, sigma=sigma, mu=mu, epsilon=epsilon, domain=(0, R),
        boundary_conditions=((wall_potential, None, None, wall_third_derivative, None),
                             (right_potential, right_slope, None, None, None)),
        initial_guess=initial_guess,
    )

bsk_problem = define_bsk_problem(
    material.coefficient, R=R, wall_potential=wall_potential,
    wall_third_derivative=wall_third_derivative,
    right_potential=right_potential, right_slope=right_slope,
    delta=delta, gamma=gamma, sigma=sigma, mu=mu,
)


## 3b. Define the linear and nonlinear known-answer problems

The reference starts with $t=x/R$ and
$u_*=1-4t/3+t^4/3$. It uses the fixed data u(0)=1, u‴(0)=0,
u(R)=u′(R)=0. At each material jump, a term proportional to
$(t-t_i)_+^3(1-t)^2$ preserves u, u′, u″, and the physical flux.
Its amplitude is supplied by `finite_segment_case`.

Only the reference fields come from the catalog. The physical RHS and its
analytic derivative are written out below:
$$
 f=J_*'+(u-u_*)+\beta(u^3-u_*^3),\qquad f_u=1+3\beta u^2.
$$
Set beta to zero for the linear case. The epsilon derivative here manufactures
the known forcing; the Galerkin solver itself does not differentiate epsilon.

In [ ]:
def define_manufactured_problem(material, *, R, delta, beta="0", initial_amplitude=None):
    reference = finite_segment_case(material, R=R, delta=delta, nonlinear=True, beta=beta)

    def f(x, u, a):
        medium = material.bind(a)
        exact = reference.u(x, a)
        delta_squared = a.scalar(delta) ** 2
        flux_dx = (
            medium.epsilon_dx(x) * (reference.derivative(x, a, 1)
                                    - delta_squared * reference.derivative(x, a, 3))
            + medium.epsilon(x) * (reference.derivative(x, a, 2)
                                   - delta_squared * reference.derivative(x, a, 4))
        )
        value = flux_dx + (u - exact)
        reaction = a.scalar(beta)
        return value if reaction == 0 else value + reaction * (u**3 - exact**3)

    def f_u(x, u, a):
        return a.scalar(1) + 3 * a.scalar(beta) * u * u

    def initial_guess(x, a):
        if initial_amplitude is not None:
            return a.scalar(initial_amplitude)
        return a.scalar(1) - x / a.scalar(R)

    problem = FluxProblem(
        delta=delta, epsilon=material.coefficient, domain=(0, R), f=f, f_u=f_u,
        boundary_conditions=(("1", None, None, "0", None),
                             ("0", "0", None, None, None)),
        initial_guess=initial_guess,
    )
    return problem, reference

linear_problem, linear_case = define_manufactured_problem(material, R=R, delta=delta, beta="0")
nonlinear_problem, nonlinear_case = define_manufactured_problem(
    material, R=R, delta=delta, beta=beta_nonlinear,
)


## 4. Run the selected problem

Set **N_selected** and **newton_atol_selected** explicitly. The latter is an
absolute target for the Euclidean norm of the algebraic residual; it is not
an absolute error bound on u. The accuracy profile selects arithmetic and
integration settings and cannot replace these two choices. Starting quadrature
scales as `2*N+4` and is checked independently by refinement.

In [ ]:
N_selected = 12
newton_atol_selected = "1e-10"
selected_problem, selected_case = {
    "BSK": (bsk_problem, None),
    "manufactured_linear": (linear_problem, linear_case),
    "manufactured_nonlinear": (nonlinear_problem, nonlinear_case),
}[model]
experiment = run_defined_problem(
    selected_problem, N=N_selected, newton_atol=newton_atol_selected,
    material=material, case=selected_case, model=model,
    accuracy=accuracy, verbosity=verbosity, numerical=numerical_settings,
)
print(report_experiment(experiment))


## 5. Inspect the fields and convergence

Dotted guides mark interfaces. Flux curves are drawn one layer at a time.
The exact transmission flux is continuous, but a finite global approximation
can have different one-sided numerical flux traces at an epsilon jump.
The Newton and integration histories are separate panels. A small Newton
residual alone does not establish spatial accuracy.

### Display controls

Edit `plot_x_min` and `plot_x_max` in the experiment settings to choose the
physical x window for profiles, material previews, and error plots. The default is `[0, R]`. `plot_x_max=None` uses each result's actual endpoint.
To view up to 50, use `plot_x_max="50"` with a physical domain reaching at least 50.
The custom and hard examples have their own display controls because their domains can differ.

These settings change drawing and sampling only. Solver inputs, stored
solutions, quadrature checks, and full-domain physical error norms stay the same.
Redraw the plotting calls using the stored results after changing the window.
`iteration_xlim` limits the Newton-iteration axis and `quadrature_xlim` limits
the quadrature-order axis; use `None` for automatic history limits.


In [ ]:
fields_figure = plot_experiment(experiment, x_min=plot_x_min, x_max=plot_x_max)
convergence_figure = plot_convergence(experiment, iteration_xlim=iteration_xlim, quadrature_xlim=quadrature_xlim)
plt.show()


## 6. Preview all six interface counts

This gallery displays material definitions only; it runs no additional solves.
Change `interface_count` in the settings cell and rerun to select a different
case. The optional sweep below solves BSK and both known-answer models.

In [ ]:
interface_counts = (1, 2, 3, 5, 7, 10)
gallery_materials = {count: segment_material(count, R=R, kind=material_style)
                     for count in interface_counts}
gallery_figure = plot_material_gallery(kind=material_style, materials=gallery_materials, x_min=plot_x_min, x_max=R if plot_x_max is None else plot_x_max)
plt.show()


## 7. Linear verification against the known answer

This check has its own explicit N and Newton target. It uses the linear
physical definition above, including genuine material jumps.

In [ ]:
N_linear = 45
newton_atol_linear = "1e-10"
linear_check = run_defined_problem(
    linear_problem, N=N_linear, newton_atol=newton_atol_linear,
    material=material, case=linear_case, model="manufactured_linear",
    accuracy=accuracy, verbosity=verbosity, numerical=numerical_settings,
)
print(report_experiment(linear_check))
linear_fields_figure = plot_experiment(linear_check, x_min=plot_x_min, x_max=plot_x_max)
linear_error_figure = plot_errors(linear_check, x_min=plot_x_min, x_max=plot_x_max)
plt.show()


### Absolute and relative errors on [0,R]

$E_{\rm abs}=\|v_N-v_*\|_{L^2(0,R)}$ and
$E_{\rm rel}(\%)=100E_{\rm abs}/\|v_*\|_{L^2(0,R)}$.
The table uses a separate refined rule split at all interfaces. A zero
reference norm gives an undefined relative error. Interface flux peaks can
remain visible even when Newton converges; inspect refinement in N next.

In [ ]:
print(report_error_table(linear_check))


## 8. Nonlinear verification against the known answer

The exact fields are the same as in the linear check. The extra cubic term
changes the nonlinear solve and its Jacobian. Similar final field errors can
therefore be expected when both runs resolve the same spatial approximation.

In [ ]:
N_nonlinear = 32
newton_atol_nonlinear = "1e-10"
nonlinear_check = run_defined_problem(
    nonlinear_problem, N=N_nonlinear, newton_atol=newton_atol_nonlinear,
    material=material, case=nonlinear_case, model="manufactured_nonlinear",
    accuracy=accuracy, verbosity=verbosity, numerical=numerical_settings,
)
print(report_experiment(nonlinear_check))
nonlinear_fields_figure = plot_experiment(nonlinear_check, x_min=plot_x_min, x_max=plot_x_max)
nonlinear_error_figure = plot_errors(nonlinear_check, x_min=plot_x_min, x_max=plot_x_max)
plt.show()


### Absolute and relative errors on [0,R]

Compare these independently integrated errors with the linear table. Tightening
Newton's target or increasing arithmetic precision alone cannot remove spatial
error from unresolved derivative jumps.

In [ ]:
print(report_error_table(nonlinear_check))


## 9. Optional: change N while keeping R fixed

Enable this cell to study the explicitly listed resolutions. It reuses the
same nonlinear problem, coefficient, precision, and absolute Newton target.
The table reports residuals alongside absolute and relative physical errors;
it does not assume a spectral convergence rate across derivative jumps.

In [ ]:
run_n_convergence = False
N_values = [6, 10, 16, 24]
newton_atol_n_convergence = "1e-10"
n_results = ()
if run_n_convergence:
    n_results = tuple(
        run_defined_problem(
            nonlinear_problem, N=current_N, newton_atol=newton_atol_n_convergence,
            material=material, case=nonlinear_case, model="manufactured_nonlinear",
            accuracy=accuracy, verbosity=verbosity, numerical=numerical_settings,
        )
        for current_N in N_values
    )
    print(report_convergence_table(n_results))


## 10. Optional: change R in the BSK problem

This is a **domain-sensitivity experiment**, separate from changing N. It
holds N_R, source parameters, boundary values, and the original material's
physical interface positions fixed; it does not regenerate scaled presets.
Every R must lie beyond the last interface. Choose a probe point inside every
interval. Verify that N_R adequately resolves each domain before interpreting
these trends. Increasing R changes the boundary-value problem; this comparison
is not a proof of convergence to the half-line solution.

In [ ]:
run_r_convergence = True
R_values = ["4", "6", "8", "16"]
N_R = 16
newton_atol_R = "1e-10"
R_probe = "1"
r_results = {}
if run_r_convergence:
    for current_R in R_values:
        domain_problem = define_bsk_problem(
            material.coefficient, R=current_R, wall_potential=wall_potential,
            wall_third_derivative=wall_third_derivative,
            right_potential=right_potential, right_slope=right_slope,
            delta=delta, gamma=gamma, sigma=sigma, mu=mu,
        )
        r_results[current_R] = run_defined_problem(
            domain_problem, N=N_R, newton_atol=newton_atol_R,
            material=material, model="BSK", accuracy=accuracy,
            verbosity=verbosity, numerical=numerical_settings,
        )
    print("R | N | Newton absolute tolerance | u(probe) | Final residual | Converged")
    for current_R, result in r_results.items():
        solution = result.solution
        print(current_R, solution.N, newton_atol_R, solution.u(R_probe),
              solution.newton_result.residual_norm, solution.converged, sep=" | ")
    r_figure = plot_domain_comparison(r_results, x_min=plot_x_min, x_max=plot_x_max)
    plt.show()


## 11. Optional: mpmath arithmetic

This independent one-interface example uses 50 digits, with its own explicit
N and absolute Newton target. A tighter integration check is also explicit.
Its spatial error can remain much larger than the residual target. The tiny
exact-space precision regressions belong to the automated solver tests.

In [ ]:
run_high_precision = True
N_high_precision = 35
newton_atol_high_precision = "1e-35"
precision_digits = 50
high_precision_result = None
if run_high_precision:
    mp_material = segment_material(1, R=R, kind="constant")
    mp_problem, mp_case = define_manufactured_problem(
        mp_material, R=R, delta=delta, beta=beta_nonlinear,
    )
    high_precision_result = run_defined_problem(
        mp_problem, N=N_high_precision, newton_atol=newton_atol_high_precision,
        material=mp_material, case=mp_case, model="manufactured_nonlinear",
        accuracy="high_precision", verbosity=verbosity,
        numerical=dict(**numerical_settings, precision=precision_digits,
                       quadrature_atol="1e-30", quadrature_rtol="1e-30"),
    )
    print(report_experiment(high_precision_result))
    print(report_error_table(high_precision_result))
    mp_figure = plot_experiment(high_precision_result, x_min=plot_x_min, x_max=plot_x_max)
    plt.show()


## 12. Optional: write an arbitrary epsilon(x)

Use the arithmetic object `a` for elementary functions and decimal constants.
Declare positive bounds, the right trace at zero, and every interface. This
BSK example needs no epsilon derivative and makes no known-answer claim.

In [ ]:
# Independent display window for the custom finite problem.
custom_plot_x_min = "0"
custom_plot_x_max = None  # Use the custom result's own domain.

run_custom_epsilon = True
N_custom = 12
newton_atol_custom = "1e-10"
custom_result = None
R = 10
if run_custom_epsilon:
    custom_interfaces = tuple(row[0] for row in segment_material(2, R=R).rows[1:])

    def custom_epsilon(x, a):
        if x < a.scalar(custom_interfaces[0]):
            return 1 + a.scalar("0.1") * a.sin(x)**2
        if x < a.scalar(custom_interfaces[1]):
            return 2 + a.scalar("0.1") * a.exp(-x / a.scalar(R))
        return a.scalar("0.8") + a.scalar("0.1") * x / a.scalar(R)

    custom_material = Coefficient(
        custom_epsilon, lower_bound="0.8", upper_bound="2.1", wall_value="1",
        interfaces=custom_interfaces,
    )
    custom_problem = define_bsk_problem(
        custom_material, R=R, wall_potential=wall_potential,
        wall_third_derivative=wall_third_derivative,
        right_potential=right_potential, right_slope=right_slope,
        delta=delta, gamma=gamma, sigma=sigma, mu=mu,
    )
    custom_result = run_defined_problem(
        custom_problem, N=N_custom, newton_atol=newton_atol_custom,
        material=custom_material, model="BSK", accuracy=accuracy,
        verbosity=verbosity, numerical=numerical_settings,
    )
    print(report_experiment(custom_result))
    custom_figure = plot_experiment(custom_result, x_min=custom_plot_x_min, x_max=custom_plot_x_max)
    plt.show()


## 13. Optional: solve all six interface counts

This cell runs **18 cases**: BSK, linear known-answer, and nonlinear known-answer
problems for each interface count. Select every N explicitly in the table.
It keeps the chosen R, physical parameters, material style, and arithmetic.
Start with standard arithmetic. No sweep runs until its switch is enabled.

In [ ]:
run_all_interfaces = True
N_by_interfaces = {1: 8, 2: 8, 3: 12, 5: 12, 7: 16, 10: 16}
newton_atol_sweep = "1e-10"
sweep_results = []
if run_all_interfaces:
    for count in interface_counts:
        sweep_material = segment_material(count, R=R, kind=material_style)
        sweep_bsk = define_bsk_problem(
            sweep_material.coefficient, R=R, wall_potential=wall_potential,
            wall_third_derivative=wall_third_derivative,
            right_potential=right_potential, right_slope=right_slope,
            delta=delta, gamma=gamma, sigma=sigma, mu=mu,
        )
        sweep_linear, reference_linear = define_manufactured_problem(sweep_material, R=R, delta=delta)
        sweep_nonlinear, reference_nonlinear = define_manufactured_problem(
            sweep_material, R=R, delta=delta, beta=beta_nonlinear,
        )
        for label, problem, reference in (
            ("BSK", sweep_bsk, None),
            ("manufactured_linear", sweep_linear, reference_linear),
            ("manufactured_nonlinear", sweep_nonlinear, reference_nonlinear),
        ):
            result = run_defined_problem(
                problem, N=N_by_interfaces[count], newton_atol=newton_atol_sweep,
                material=sweep_material, case=reference, model=label,
                accuracy=accuracy, verbosity=verbosity, numerical=numerical_settings,
            )
            sweep_results.append(result)
            print(report_experiment(result))
            if reference is not None:
                print(report_error_table(result))


## 14. Optional: watch homotopy recover a difficult solve

This example has its own material, interval, resolution, starting field, and
Newton target. It does not inherit changes to the main experiment. The same
problem is run three ways: a short direct Newton budget, continuation with the
same per-stage budget, and a longer direct Newton control.

Continuation first attempts the physical problem. If that fails, it verifies
a screened linear reference and follows
$R_h(z;q)=(1-q)R_h^{\rm physical}(z)+qR_h^{\rm reference}(z)$
from q=1 to the physical endpoint q=0. Accepted states survive rejected trials.
Both the finite domain and all boundary data are retained.

In [ ]:
# Independent display window: the hard problem has its own R_hard.
hard_plot_x_min = "0"
hard_plot_x_max = None  # Use the hard result's own domain.

run_homotopy_demo = True
R_hard = "2"
N_hard = 6
newton_atol_hard = "1e-10"
hard_delta = "2"
hard_beta = "2"
hard_initial_amplitude = "30"
hard_stage_iterations = 5
hard_control_iterations = 40
hard_use_mpmath = False
hard_precision = 30
hard_material = LayeredMaterial((("0", "1"), ("1", "2")))
hard_problem, hard_case = define_manufactured_problem(
    hard_material, R=R_hard, delta=hard_delta, beta=hard_beta,
    initial_amplitude=hard_initial_amplitude,
)
hard_continuation = ContinuationOptions(
    reference=ScreenedLinearReference(strength="1"), initial_step="0.5",
    max_step="0.5", min_step="1e-6", growth="1.5",
    max_stages=40, max_total_iterations=150,
)


In [ ]:
hard_results = {}
if run_homotopy_demo:
    for label, globalization, iteration_limit in (
        ("direct_limited", False, hard_stage_iterations),
        ("homotopy", True, hard_stage_iterations),
        ("direct_control", False, hard_control_iterations),
    ):
        hard_results[label] = run_defined_problem(
            hard_problem, N=N_hard, newton_atol=newton_atol_hard,
            material=hard_material, case=hard_case, model="hard finite nonlinear case",
            accuracy="standard", verbosity=verbosity,
            numerical=dict(use_mpmath=hard_use_mpmath, precision=hard_precision,
                           max_iterations=iteration_limit, use_globalization=globalization,
                           continuation=hard_continuation, newton_fallback=None,
                           quadrature_order=2*N_hard+4, quadrature_atol="1e-8",
                           quadrature_rtol="1e-8", max_refinements=2,
                           progress_callback=progress_callback, progress_interval=progress_interval),
        )
    print(report_homotopy_comparison(hard_results))


### Check the accepted stages and the physical endpoint

The table includes reference verification and accepted/rejected continuation
stages. The final residual is a physical residual only at q=0. If edited
parameters make direct Newton succeed, this is no longer a recovery example;
the result is reported as such. The longer direct control shows that a short
stage budget can cause failure; recovery does not prove homotopy is necessary
or faster in general.

In [ ]:
if run_homotopy_demo:
    recovered = hard_results["homotopy"]
    print(report_homotopy_stages(recovered))
    for label, result in hard_results.items():
        print(label)
        print(report_error_table(result))
    direct_failed = not hard_results["direct_limited"].solution.converged
    hard_stage_records = homotopy_stage_rows(recovered)
    reference_used = any(row["kind"] == "reference" and row["accepted"]
                         for row in hard_stage_records)
    positive_stage_accepted = any(row["kind"] == "continuation" and row["accepted"]
                                  and 0 < row["q"] < 1 for row in hard_stage_records)
    physical_reached = recovered.solution.converged and recovered.solution.newton_result.homotopy == 0
    print("Short direct solve failed:", direct_failed)
    print("Verified reference recovery demonstrated:",
          direct_failed and reference_used and positive_stage_accepted and physical_reached)
    print("Longer direct control converged:", hard_results["direct_control"].solution.converged)


In [ ]:
hard_figures = {}
if run_homotopy_demo:
    hard_figures["parameter-path"] = plot_homotopy_path(hard_results["homotopy"])
    hard_figures["residuals"] = plot_convergence(hard_results['homotopy'], iteration_xlim=iteration_xlim, quadrature_xlim=quadrature_xlim)
    hard_figures["fields"] = plot_homotopy_comparison(hard_results, x_min=hard_plot_x_min, x_max=hard_plot_x_max)
    hard_figures["errors"] = plot_errors(hard_results['homotopy'], x_min=hard_plot_x_min, x_max=hard_plot_x_max)
    plt.show()


## 15. Optional: save the experiment

The selected result can be exported as `settings.json`, a typed `solution.json`,
native-precision `profiles.csv` covering [0,R], and PDF figures. The settings
retain R, N, Newton tolerance, arithmetic, and quadrature checks. Executable
source/material callbacks are not serialized: keep the problem-definition
cells with your data. Set `OVERSCREENING_OUTPUT_DIR` to choose another location.

In [ ]:
save_results = False
output_directory = Path(os.environ.get("OVERSCREENING_OUTPUT_DIR", "notebook-output")) / "segment-galerkin"
saved_files = {}
if save_results:
    saved_files = export_experiment(
        experiment, output_directory,
        figures={"material": material_figure, "fields": fields_figure,
                 "convergence": convergence_figure},
    )
    for label, path in saved_files.items():
        print(label, path)


## Reading the results

Use the physical field/error plots, the residual and integration checks, and
the separate N/R studies together. Exact transmission flux continuity is a
property of the reference solution; finite global polynomial fields need not
have matching pointwise interface fluxes. Every result records its chosen N
and Newton absolute tolerance.

The finite trial/test spaces differ (a Petrov formulation). The half-line
stability proof does not automatically establish stability of this finite
method. Implementation checks and mathematical stability analysis have
different roles. See [the finite-interval guide](../../docs/finite_galerkin.md).